# A simulated market session

The whole pipeline, end to end: a parametrization read off disk, synthetic order flow
generated from it, a book folded through that flow, and the statistics read off the book.

The statistics come last and are the smallest part. What the strand is actually about is
the middle: **order flow arrives and queues respond**, and everything else is a summary of
that.

Two things recur and are worth watching for:

- **every series is a step line.** The book holds each state until the next message, so a
  sloped segment would draw states the book never had;
- **a level index means two different things.** The notes count positions on the price
  grid; a LOBSTER file counts prices that carry volume. See
  [`grid-levels-and-lobster-levels.md`](../documentation/grid-levels-and-lobster-levels.md).

In [1]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from unito26.lob import config, frames
from unito26.lob.benchmark import REFERENCE_PRICE
from unito26.lob.messages import BUY, SELL, GridDepth, ReportedDepth
from unito26.lob.orderbook import AXIS_B_VARIANTS, AggregateBook, TickArrayBook
from unito26.lob.replay import MarketSession
from unito26.lob.simulate import OrderFlowSimulator

BID, ASK = "#2a78d6", "#eb6834"
SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"

pio.templates["unito26"] = go.layout.Template(layout=dict(
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(color=INK, size=12),
    xaxis=dict(gridcolor=GRID, linecolor="#c3c2b7", zeroline=False, tickfont=dict(color=MUTED)),
    yaxis=dict(gridcolor=GRID, linecolor="#c3c2b7", zeroline=False, tickfont=dict(color=MUTED)),
    hovermode="x unified", colorway=[BID, ASK],
))
pio.templates.default = "unito26"

def steps(x, y, name, color, **kwargs):
    line = dict(color=color, width=2) | kwargs.pop("line", {})
    return go.Scatter(x=x, y=y, name=name, mode="lines", line=line,
                      line_shape="hv", connectgaps=False, **kwargs)

## 1. The parametrization

`config` ships *examples*, not defaults, and ships them frozen as serialized frames. The
notebook demonstrates loading a specification; nothing here is computed at import time.

In [2]:
flow = config.example_order_flow_params()
marks = config.example_mark_params()

print(f"branching ratio      {flow.branching_ratio:.4f}")
print(f"stationary intensity {np.round(flow.stationary_intensity(), 3)}  events/s per type")
print(f"marks                {marks}")
frames.hawkes_params_to_frame(flow).head(8)

branching ratio      0.8000
stationary intensity [2.203 2.203 7.53  7.53  5.361 5.361]  events/s per type
marks                MarkParams(depth_decay=0.45, mean_log_size=4.0, sigma_log_size=0.8, lot=10)


,Component,Cause,BaseIntensity,Kernel,Decay
0,0,0,0.3,24.186291,60.0
1,0,1,NaN,5.374731,60.0
2,0,2,NaN,1.343683,60.0
3,0,3,NaN,1.343683,60.0
4,0,4,NaN,2.687366,60.0
5,0,5,NaN,2.687366,60.0
6,1,0,NaN,5.374731,60.0
7,1,1,0.3,24.186291,60.0


Read the branching ratio rather than trusting it: at 0.8, four events in five are
triggered by another event rather than arriving on their own. The excitation is a
$6\times6$ matrix in long form, one row per (excited, exciting) pair — the baseline is a
vector, so it sits on the diagonal and is null off it.

## 2. The flow, and warming the book up

A book that starts empty is unrepresentative for a while: the first orders have nothing to
trade against. So the simulator runs into it before anything is recorded.

In [3]:
simulator = OrderFlowSimulator(flow, marks, REFERENCE_PRICE, rng=11)
book = AggregateBook()
simulator.warm_up(book, horizon=60.0)
opening = book.copy()          # the state the recorded session starts from
print("after warm-up:", book)

# The stream is a generator that reads the book as it currently stands, so the driver has
# to apply each message before the next is computed.  Consuming it without applying would
# quote every order against the frozen warm-up state.
messages = []
for message in simulator.stream(book, horizon=600.0):
    book.apply(message)
    messages.append(message)
print(f"{len(messages)} messages over 600s")

after warm-up: AggregateBook(bid=10002x560, ask=10003x4070, levels=7/10)


16663 messages over 600s


## 3. Folding the session

This is the centre of the pipeline: order flow in, book states out. `MarketSession` records
the top `reported_depth` **occupied** levels after every message, and reads the statistics
off the book as it goes.

The two depths are separate arguments because they are separate quantities —
`reported_depth` counts occupied levels, `imbalance_levels` counts grid positions.

In [4]:
DEPTH = ReportedDepth(10)
LEVELS = tuple(GridDepth(n) for n in (1, 2, 3, 5, 10))
PRICE_UNIT = 100   # LOBSTER quotes dollars x 10000, so a penny tick is 100

session = MarketSession.from_occupied_levels(
    opening.copy(), messages, DEPTH, LEVELS, PRICE_UNIT, record_deltas=True
)
print(f"{len(session.lobster_book)} rows, {len(session.level_deltas)} level changes")

# The statistics and the timings use the whole session; the figures draw a window of it.
# Sixteen thousand step points on forty traces is neither readable nor small.
WINDOW = slice(0, 1500)
session.lobster_book.iloc[:5, :8]

16663 rows, 16761 level changes


,AskPrice1,AskSize1,BidPrice1,BidSize1,AskPrice2,AskSize2,BidPrice2,BidSize2
TimeStamp,,,,,,,,
60.886658,1000300,4070,1000200,560,1000400,3160,1000100,1080
60.897390,1000300,4070,1000200,510,1000400,3160,1000100,1080
60.901759,1000300,4070,1000200,510,1000400,3160,1000100,1130
61.097346,1000300,4070,1000200,510,1000400,3140,1000100,1130
61.110965,1000300,4070,1000200,510,1000400,3140,1000100,1130


## 4. What came out is what a vendor sells you

`lobster_book` is exactly a LOBSTER orderbook file: $4 \times D$ integer columns, ask price
and size then bid price and size, repeated per level — and **no timestamp**, which in a
real file lives in the row-aligned message file. Time is on the index here.

A side holding fewer than $D$ occupied levels is padded, and the two sentinels have
*opposite signs*: `-9999999999` on the bid, `+9999999999` on the ask. A filter written for
one lets the other straight through.

In [5]:
padded_ask = (session.lobster_book[f"AskPrice{DEPTH}"] == frames.ASK_PADDING).mean()
padded_bid = (session.lobster_book[f"BidPrice{DEPTH}"] == frames.BID_PADDING).mean()
print(f"rows padded at level {DEPTH}:  ask {padded_ask:.1%}   bid {padded_bid:.1%}")

# It validates against the schema, which is what makes "LOBSTER compatible" a fact.
frames.lobster_book_schema(DEPTH).validate(session.lobster_book).shape

rows padded at level 10:  ask 21.2%   bid 83.8%


(16663, 40)

## 5. The statistics, twice

Route 1 read them off the evolving book, message by message. Route 2 recomputes them from
the finished frame alone, vectorized, touching no book.

They agree everywhere except on $I^n$, and there only where the frame's reported levels do
not span the grid window — which is a column, not a surprise.

In [6]:
def timed(label):
    def wrap(fn):
        start = time.perf_counter()
        result = fn()
        print(f"{label:38s} {time.perf_counter() - start:7.3f}s")
        return result
    return wrap

from_frame = timed("route 2: vectorized over the frame")(session.stats_from_frame)
from_book = timed("route 1: read off the book (already done)")(lambda: session.stats)

expected = session.stats.copy()
for n in LEVELS:
    expected.loc[~expected[f"QueueImbalance{n}Covered"].astype(bool), f"QueueImbalance{n}"] = np.nan
pd.testing.assert_frame_equal(expected, from_frame, check_dtype=False)
print("\nthe two routes agree")
session.stats[["Spread", "MidPrice", "MicroPrice", "QueueImbalance1", "AskLargestGap"]].head()

route 2: vectorized over the frame       0.013s
route 1: read off the book (already done)   0.000s

the two routes agree


,Spread,MidPrice,MicroPrice,QueueImbalance1,AskLargestGap
TimeStamp,,,,,
60.886658,1.0,10002.5,10002.120950,-0.758099,0.0
60.897390,1.0,10002.5,10002.111354,-0.777293,0.0
60.901759,1.0,10002.5,10002.111354,-0.777293,0.0
61.097346,1.0,10002.5,10002.111354,-0.777293,0.0
61.110965,1.0,10002.5,10002.111354,-0.777293,0.0


## 6. Queue evolution

The figure the strand is about. Pick a level from the dropdown: the **top panel** is the
volume resting at that level on each side, the **bottom panel** the price of that level.

Splitting by quantity rather than by side is what makes it readable — each panel compares
the two sides directly, and they share a time axis, so a queue draining above and a price
stepping below are visibly the same event. Colour encodes side and nothing else, so it
stays the same in both panels.

Watch the price panel: the two lines can never cross, and the distance between them is the
$n$-level spread, widening as you go deeper. Where a side has fewer than $n$ occupied
levels the line **breaks** — that level did not exist then. Drawing the padding literally
would put one point at $10^{10}$ and destroy the axis.

In [7]:
def level_figure(session, window):
    book, depth = session.lobster_book.iloc[window], session.reported_depth
    time_axis = book.index

    def series(side, kind, level, padding):
        price = book[f"{side}Price{level}"].to_numpy(dtype=float)
        values = book[f"{side}{kind}{level}"].to_numpy(dtype=float)
        # A padded level is absent, not zero-priced: NaN so the step line breaks.
        return np.where(price == padding, np.nan, values / (PRICE_UNIT if kind == "Price" else 1))

    figure = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                           subplot_titles=("Volume at the level", "Price of the level"))
    for level in range(1, depth + 1):
        visible = level == 1
        for row, kind in ((1, "Size"), (2, "Price")):
            for side, colour, padding in (("Bid", BID, frames.BID_PADDING), ("Ask", ASK, frames.ASK_PADDING)):
                figure.add_trace(
                    steps(time_axis, series(side, kind, level, padding), side, colour,
                          visible=visible, legendgroup=side, showlegend=(row == 1 and level == 1)),
                    row=row, col=1,
                )

    # One button per level; the mask spans every trace in the figure, four of them true.
    per_level = 4
    buttons = [
        dict(label=f"level {level}", method="update",
             args=[{"visible": [i // per_level == level - 1 for i in range(per_level * depth)]},
                   {"title": f"Occupied level {level}"}])
        for level in range(1, depth + 1)
    ]
    figure.update_layout(
        title="Occupied level 1",
        updatemenus=[dict(buttons=buttons, active=0, x=1.0, xanchor="right", y=1.14,
                          yanchor="top", showactive=True)],
        height=620, margin=dict(t=110, r=20),
        legend=dict(orientation="h", y=1.06, x=0),
    )
    figure.update_yaxes(title_text="shares", row=1, col=1)
    figure.update_yaxes(title_text="ticks", row=2, col=1)
    figure.update_xaxes(title_text="session time (s)", row=2, col=1)
    return figure

level_figure(session, WINDOW)

The dropdown selects a **reported** level — the $n$-th price carrying volume — because
that is what the frame holds. On a book with holes that is not the $n$-th grid position.

## 7. Mid-price and micro-price

$P^\mu$ rides inside the spread and leans toward the *thin* side, since it weights each
price by the volume resting on the **other** one. Written out, that is exactly

$$P^\mu = P^m + \frac{\phi}{2} I^1,$$

so the micro-price is the mid displaced by the imbalance, in units of the half-spread.

In [8]:
stats = session.stats.iloc[WINDOW]
times = stats.index
figure = go.Figure([
    steps(times, stats["MidPrice"] + stats["Spread"] / 2, "best ask", ASK,
          line=dict(color=ASK, width=1, dash="dot"), showlegend=True),
    steps(times, stats["MidPrice"] - stats["Spread"] / 2, "best bid", BID,
          line=dict(color=BID, width=1, dash="dot"), showlegend=True),
    steps(times, stats["MidPrice"], "mid-price", MUTED),
    steps(times, stats["MicroPrice"], "micro-price", "#4a3aa7"),
])
figure.update_layout(title="The micro-price inside the spread", height=420,
                     xaxis_title="session time (s)", yaxis_title="ticks",
                     legend=dict(orientation="h", y=1.08, x=0))
figure

## 8. The spread, and how gappy the book is

The gap statistics say how the occupied levels are distributed: `LargestGap` is the longest
run of empty grid positions between two levels that carry volume, over the reported span.
A book that is dense near the touch and sparse below it will show it here.

In [9]:
figure = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                       subplot_titles=("Spread", "Largest gap between occupied levels"))
figure.add_trace(steps(times, stats["Spread"], "spread", MUTED, showlegend=False), row=1, col=1)
figure.add_trace(steps(times, stats["BidLargestGap"], "Bid", BID, legendgroup="Bid"), row=2, col=1)
figure.add_trace(steps(times, stats["AskLargestGap"], "Ask", ASK, legendgroup="Ask"), row=2, col=1)
figure.update_layout(height=520, legend=dict(orientation="h", y=1.06, x=0))
figure.update_yaxes(title_text="ticks", row=1, col=1)
figure.update_yaxes(title_text="empty positions", row=2, col=1)
figure.update_xaxes(title_text="session time (s)", row=2, col=1)
figure

## 9. The imbalance, and its impostor

$I^n$ sums volume over the first $n$ positions **on the price grid**. The obvious way to
compute it from a file — sum the first $n$ size *columns* — is a different statistic, and
they diverge exactly when the book has holes inside the window.

Neither version errors. Both stay in $[-1,1]$ and move plausibly with the market.

In [10]:
def imbalance_by_column(session, n):
    book = session.lobster_book
    ask = sum(book[f"AskSize{k}"] for k in range(1, n + 1))
    bid = sum(book[f"BidSize{k}"] for k in range(1, n + 1))
    return (bid - ask) / (bid + ask)

N = GridDepth(5)
by_column = imbalance_by_column(session, N)
by_price = session.stats_from_frame()[f"QueueImbalance{N}"]

figure = go.Figure([
    steps(times, by_column.iloc[WINDOW], f"first {N} columns (wrong)", ASK),
    steps(times, by_price.iloc[WINDOW], f"first {N} grid positions", BID),
])
figure.update_layout(title=f"I^{N}: selecting by price against slicing by column",
                     height=420, xaxis_title="session time (s)", yaxis_title="imbalance",
                     legend=dict(orientation="h", y=1.08, x=0))
figure.show()

difference = (by_column - by_price).abs()
print(f"they differ on {(difference > 1e-9).mean():.1%} of rows; largest gap {difference.max():.3f}")

they differ on 23.1% of rows; largest gap 1.576


## 10. How deep a file do you need?

The price mask can only reach as far as the reported levels span. Write the **grid span**
of a side as the number of grid positions between its best and its deepest reported level;
$I^n$ is recoverable when $n$ is within that span **on both sides**.

So the answer is not a property of the code. It is a property of the market: a book with no
holes answers a deep $I^n$ from a shallow file, a sparse one does not.

In [11]:
depths = [1, 2, 3, 5, 10]
levels = [1, 2, 3, 5, 10]
coverage = pd.DataFrame(index=pd.Index(levels, name="n"), columns=depths, dtype=float)
for depth in depths:
    trial = MarketSession.from_occupied_levels(
        AggregateBook(), messages, ReportedDepth(depth),
        tuple(GridDepth(n) for n in levels), PRICE_UNIT,
    ).stats_from_frame()
    for n in levels:
        coverage.loc[n, depth] = 1.0 - trial[f"QueueImbalance{n}Covered"].mean()

figure = go.Figure()
for n in levels:
    figure.add_trace(go.Scatter(x=depths, y=coverage.loc[n], mode="lines+markers", name=f"I^{n}",
                                line=dict(width=2), marker=dict(size=8)))
figure.update_layout(title="Fraction of the session where I^n is NOT recoverable from the file",
                     xaxis_title="reported depth of the file", yaxis_title="fraction of rows",
                     height=420, yaxis_tickformat=".0%", hovermode="x unified")
figure.show()
coverage.round(3)

,1,2,3,5,10
n,,,,,
1,0.0,0.000,0.000,0.000,0.0
2,1.0,0.000,0.000,0.000,0.0
3,1.0,0.997,0.000,0.000,0.0
5,1.0,0.998,0.997,0.000,0.0
10,1.0,0.999,0.998,0.996,0.0


## 11. Three ways to record the same session

`from_top_of_book` reads the four touch properties, `from_occupied_levels` asks for the top
$D$ levels, `from_level_deltas` records only what each message changed and rebuilds. They
must produce identical sessions, so the only question is which is faster — and that depends
on the book underneath, which is the ladder's lesson arriving from a new direction.

`from_level_deltas` is expected to lose: deltas are cheap to **store**, and reconstruction
needs full book state at every step, so it is a second fold.

In [12]:
OPENING_BIDS, OPENING_ASKS = dict(opening.levels_map(BUY)), dict(opening.levels_map(SELL))
SPAN = [m.price for m in messages if m.price < 10 ** 9] + list(OPENING_BIDS) + list(OPENING_ASKS)

# Every variant starts from the same warmed state, sized for the whole run.
def fresh(book_cls):
    book = book_cls.for_prices(SPAN)
    for direction, levels in ((BUY, OPENING_BIDS), (SELL, OPENING_ASKS)):
        for price, volume in levels.items():
            book.set_volume(direction, price, volume)
    return book

sample = messages[:6000]
rows = []
for book_cls in AXIS_B_VARIANTS:
    timings = {}
    for label, call in (
        ("top_of_book", lambda c=book_cls: MarketSession.from_top_of_book(fresh(c), sample, LEVELS, PRICE_UNIT)),
        ("occupied(1)", lambda c=book_cls: MarketSession.from_occupied_levels(fresh(c), sample, ReportedDepth(1), LEVELS, PRICE_UNIT)),
        ("occupied(10)", lambda c=book_cls: MarketSession.from_occupied_levels(fresh(c), sample, ReportedDepth(10), LEVELS, PRICE_UNIT)),
        ("level_deltas(10)", lambda c=book_cls: MarketSession.from_level_deltas(fresh(c), sample, ReportedDepth(10), LEVELS, PRICE_UNIT)),
    ):
        start = time.perf_counter()
        call()
        timings[label] = time.perf_counter() - start
    rows.append(pd.Series(timings, name=book_cls.__name__))

pd.DataFrame(rows).round(3)

,top_of_book,occupied(1),occupied(10),level_deltas(10)
AggregateBook,0.216,0.217,0.593,0.581
CachedBestBook,0.151,0.153,0.507,0.500
HeapBook,0.189,0.191,0.550,0.544
BitmapBook,0.280,0.279,0.627,0.529
TickArrayBook,0.215,0.214,1.044,0.445


## 12. The gap statistics across the ladder

The dict-backed books walk sorted keys; the bitmap-backed ones read the answer off an
integer, using `measure_largest_binary_gap` and `count_binary_gaps`. Whether that is worth
anything depends on how many levels are occupied — the same conditional the best-price
ladder produced.

In [13]:
def gap_pass(book, reported_depth):
    for direction in (BUY, SELL):
        book.largest_gap_size_between_non_empty_levels(direction, reported_depth)
        book.gap_count(direction, reported_depth)

rows = []
for regime, mark_params in (("shallow", config.shallow_mark_params()), ("deep", config.deep_mark_params())):
    stream = OrderFlowSimulator(flow, mark_params, REFERENCE_PRICE, rng=3)
    seed_book = AggregateBook()
    stream.warm_up(seed_book, horizon=120.0)
    occupied = len(seed_book.levels_map(BUY)) + len(seed_book.levels_map(SELL))
    timings = {"occupied levels": occupied}
    for book_cls in AXIS_B_VARIANTS:
        book = book_cls.from_levels(dict(seed_book.levels_map(BUY)), dict(seed_book.levels_map(SELL)))
        start = time.perf_counter()
        for _ in range(2000):
            gap_pass(book, DEPTH)
        timings[book_cls.__name__] = round(time.perf_counter() - start, 3)
    rows.append(pd.Series(timings, name=regime))

pd.DataFrame(rows)

,occupied levels,AggregateBook,CachedBestBook,HeapBook,BitmapBook,TickArrayBook
shallow,18.0,0.027,0.023,0.024,0.014,0.020
deep,241.0,0.152,0.135,0.134,0.069,0.026


Whatever these numbers say goes into `dev-context/market-microstructure.md` beside the
strand's other measured findings — including where they contradict the reasoning above.
That has happened before on this ladder, and it is the more useful outcome.